# Day 10 Solutions: Decision Problems (instructor copy)

Each solution runs with the first test row and ends with a `print(... == expected)` check. The complete expected
tables are given in markdown: use them to check students' tables quickly.

## Exercise 1: Week 2 review

| | Output | Common wrong answer |
|-|--------|---------------------|
| a | `25.0 20.0 [25.0, 20.0]` | `min` = 19.0 (forgot the change) |
| b | `['red', 'blue', 'green'] 1 blue, green, red` | both greens removed |
| c | `2 3 TBC` | `len(team)` = 3 or 4 (counting the players) |
| d | `4 ['Omar'] ['Ken']` | `5` for the union (Omar counted twice) |
| e | `Sold out` | `In stock` (0 counts as `False`) |

In [2]:
temps = [21.5, 25.0, 19.0, 23.5]
temps[2] = 20.0
print(max(temps), min(temps), temps[1:3])

25.0 20.0 [25.0, 20.0]


In [3]:
words = "red green blue green".split()
words.remove("green")
print(words, words.count("green"), ", ".join(sorted(words)))

['red', 'blue', 'green'] 1 blue, green, red


In [4]:
team = {"captain": "Zara", "players": ["Ken", "Mei"]}
team["players"].append("Diego")
print(len(team), len(team["players"]), team.get("coach", "TBC"))

2 3 TBC


In [5]:
monday = {"Asha", "Omar", "Mei"}
tuesday = {"Omar", "Ken"}
print(len(monday | tuesday), sorted(monday & tuesday), sorted(tuesday - monday))

4 ['Omar'] ['Ken']


In [6]:
stock = 0
if stock:
    print("In stock")
elif stock == 0:
    print("Sold out")
else:
    print("Error")

Sold out


## Exercise 2: Ticket-price calculator

Expected column:

| # | age | day | is_student | Expected |
|---|-----|-----|------------|----------|
| 1 | 2 | Sat | False | Ticket price: 0.00 |
| 2 | 3 | Mon | False | Ticket price: 6.00 |
| 3 | 12 | Sun | False | Ticket price: 7.50 |
| 4 | 13 | Mon | False | Ticket price: 12.00 |
| 5 | 20 | Sat | True | Ticket price: 12.00 |
| 6 | 20 | Sat | False | Ticket price: 15.00 |
| 7 | 64 | Tue | False | Ticket price: 12.00 |
| 8 | 65 | Tue | False | Ticket price: 7.20 |
| 9 | 70 | Sun | True | Ticket price: 9.00 (senior discount wins; only one applies) |
| 10 | 30 | Funday | False | Invalid day |
| 11 | -1 | Mon | False | Invalid age |

In [7]:
age = 2
day = "Sat"
is_student = False

base_prices = {"Mon": 12.0, "Tue": 12.0, "Wed": 12.0, "Thu": 12.0, "Fri": 12.0, "Sat": 15.0, "Sun": 15.0}

if day not in base_prices:
    message = "Invalid day"
elif age < 0 or age > 120:
    message = "Invalid age"
else:
    price = base_prices[day]
    if age < 3:
        price = 0.0
    elif age <= 12:
        price = price * 0.5
    elif age >= 65:
        price = price * 0.6
    elif is_student:
        price = price * 0.8
    message = f"Ticket price: {price:.2f}"

print(message)
print(message == "Ticket price: 0.00")

Ticket price: 0.00
True


Reasoning to discuss:
- The discounts are **one `elif` chain** because only one may apply. The validation checks come first so the
  pricing code never sees bad data.
- `elif age <= 12` does not need `age >= 3`: the branch above already handled `age < 3`.
- "40% off" means paying 60%: `price * 0.6`. A common wrong answer is `price * 0.4`. Row 8 catches it.
- Storing the result in `message` and printing once makes the final check easy.

Alternative for the base price, with lists:

In [8]:
weekdays = ["Mon", "Tue", "Wed", "Thu", "Fri"]
weekend = ["Sat", "Sun"]
day = "Sun"

if day in weekdays:
    base = 12.0
elif day in weekend:
    base = 15.0
else:
    base = 0.0
    print("Invalid day")
print(base == 15.0)

True


The dictionary version is shorter because the lookup *is* the validity check (`day not in base_prices`).

## Exercise 3: Find the bugs with tests

With the original code, rows **1**, **6** and **8** fail (actual: 2.00, 5.00, 2.50).

- **Bug 1:** `minutes < 30` should be `minutes <= 30` ("up to and including 30"). Found by row 1, the boundary.
- **Bug 2:** `elif minutes > 120` catches **every** value above 120, so the `else` (9.00) can never run. Found by
  rows 6 and 8. The fix is `elif minutes <= 300`.

Rows 2–5, 7 and 9 pass with the buggy code, so a student who tests only "normal" values sees no problem at all.

In [9]:
minutes = 30
is_electric = False

if minutes < 0:
    message = "Invalid time"
else:
    if minutes <= 30:
        fee = 0.0
    elif minutes <= 120:
        fee = 2.0
    elif minutes <= 300:
        fee = 5.0
    else:
        fee = 9.0
    if is_electric:
        fee = fee / 2
    message = f"Parking fee: {fee:.2f}"

print(message)
print(message == "Parking fee: 0.00")

Parking fee: 0.00
True


The electric discount is a separate `if` (not part of the chain) because it applies **on top of** any fee.

## Exercise 4: Password strength checker

Full expected output:

| # | password | Output |
|---|----------|--------|
| 1 | `zara2026` | Score: 3/5 → medium · Missing: an uppercase letter, not containing the username |
| 2 | `tulip` | Score: 2/5 → weak · Missing: at least 8 characters, an uppercase letter, a digit |
| 3 | `Blue-Kite-7` | Score: 5/5 → strong · Missing: nothing |
| 4 | `password123` | Rejected: too common |
| 5 | `ZARA-is-9-cool` | Score: 4/5 → medium · Missing: not containing the username |

In [10]:
common_passwords = ["password", "password123", "12345678", "qwerty123", "letmein"]
username = "zara"
password = "zara2026"

score = 0
missing = []

if password in common_passwords:
    print("Rejected: too common")
else:
    if len(password) >= 8:
        score += 1
    else:
        missing.append("at least 8 characters")
    if password.lower() != password:
        score += 1
    else:
        missing.append("an uppercase letter")
    if password.upper() != password:
        score += 1
    else:
        missing.append("a lowercase letter")
    if set(password) & set("0123456789"):
        score += 1
    else:
        missing.append("a digit")
    if username.lower() not in password.lower():
        score += 1
    else:
        missing.append("not containing the username")

    if score == 5:
        strength = "strong"
    elif score >= 3:
        strength = "medium"
    else:
        strength = "weak"
    print(f"Score: {score}/5 → {strength}")
    if missing:
        print("Missing: " + ", ".join(missing))
    else:
        print("Missing: nothing")

print(score == 3)
print(missing == ["an uppercase letter", "not containing the username"])

Score: 3/5 → medium
Missing: an uppercase letter, not containing the username
True
True


Teaching points:
- Five **separate** `if`s, because every check is scored independently. Students who use `elif` here stop after
  the first passing check (a good bug to discuss).
- "Has an uppercase letter" = lowercasing the password changes it. Students find this trick surprising; let them
  try `"abc".lower() != "abc"` to convince themselves.
- `set(password) & set("0123456789")` is a set of the digits in the password; an empty set counts as `False`.
- `score += 1` is the **accumulator** idea that loops will use on Day 11.
- `if missing:` uses truthiness (Day 09) to choose between a list and `nothing`.
- Never store or print real passwords; these are made-up examples. A real system would also hash passwords
  (out of scope).

## Exercise 5 (challenge): Course enrolment desk

A good test table (each row starts from the original data):

| # | request | completed | Expected |
|---|---------|-----------|----------|
| 1 | `"cooking"` | `{"python"}` | No such course: cooking |
| 2 | `"Python"` | `{"python"}` | You have already completed python. |
| 3 | `"ML"` | `{"python"}` | Missing prerequisites: statistics |
| 4 | `"ml"` | `{"python", "statistics"}` | ml is full. You are number 1 on the waiting list. |
| 5 | `"  Data Viz "` | `{"python"}` | Enrolled in data viz. Seats left: 2 |

In [11]:
courses = {
    "python": {"seats": 2, "prerequisites": set()},
    "statistics": {"seats": 1, "prerequisites": set()},
    "ml": {"seats": 0, "prerequisites": {"python", "statistics"}},
    "data viz": {"seats": 3, "prerequisites": {"python"}},
}
completed = {"python"}      # courses this student has already finished
waiting_list = []
request = "  Data Viz "     # pretend the student typed this

name = request.strip().lower()
if name not in courses:
    message = f"No such course: {name}"
elif name in completed:
    message = f"You have already completed {name}."
elif courses[name]["prerequisites"] - completed:
    still_needed = sorted(courses[name]["prerequisites"] - completed)
    message = "Missing prerequisites: " + ", ".join(still_needed)
elif courses[name]["seats"] == 0:
    waiting_list.append(name)
    message = f"{name} is full. You are number {len(waiting_list)} on the waiting list."
else:
    courses[name]["seats"] -= 1
    message = f"Enrolled in {name}. Seats left: {courses[name]['seats']}"

print(message)
print(message == "Enrolled in data viz. Seats left: 2")

Enrolled in data viz. Seats left: 2
True


- **Why rule 3 before rule 4:** a student who is not allowed to take the course should not be put on its waiting
  list. Order in an `elif` chain encodes the business rules.
- `prerequisites - completed` is the set of courses still needed; an empty set means "all done" (truthiness).
- Common wrong answer: checking `courses[name]` before `name in courses` → `KeyError` for row 1.